# YOLO26n v2 su Apple Silicon

Questo notebook fine-tuna il checkpoint COCO del repository con MPS, crea uno split train/validation deterministico senza modificare il dataset sorgente, misura il modello migliore, esporta ONNX e prova l'intero runtime Phenocam.

La validation è ricavata dal dataset pubblico di training: misura il confronto interno v2, ma **non** sostituisce la validation operativa su immagini interne, che resta separata e pending. La testata resta di 80 classi COCO perché è un invariante del runtime; soltanto le sei classi operative hanno ground truth in questo dataset.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import hashlib
import json
import shutil

import torch
import yaml
from ultralytics import YOLO

ROOT = Path.cwd().resolve()
SOURCE = ROOT / "dataset" / "training-dataset"
BASE_MODEL = ROOT / "models" / "yolo26n.pt"
WORK = ROOT / "output" / "training-v2"
RUNS = WORK / "runs"
V2_PT = ROOT / "models" / "yolo26n-v2.pt"
V2_ONNX = ROOT / "models" / "yolo26n-v2.onnx"

EPOCHS = 30
IMAGE_SIZE = 640
BATCH = 8
SEED = 42
VALIDATION_FRACTION = 0.20
DEVICE = "mps"

if not (ROOT / "phenocam").is_dir() or not BASE_MODEL.is_file():
    raise RuntimeError("Avvia il notebook dalla radice del repository")
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS non è disponibile: usa Python arm64 con PyTorch per Apple Silicon")
WORK.mkdir(parents=True, exist_ok=True)
print({"torch": torch.__version__, "device": DEVICE, "epochs": EPOCHS, "batch": BATCH})

## Split e controlli

Il gruppo di deduplicazione/sequence è indivisibile. Il 20% dei frame viene scelto ordinando i gruppi con un hash stabile del seed; file, label, coordinate e inventario classi sono controllati prima di scrivere le liste assolute usate da Ultralytics.

In [ ]:
manifest = SOURCE / "metadata" / "source-images.csv"
required = {"image_id", "group_id", "source_dataset", "polarity", "primary_stratum", "image_path", "label_path"}
with manifest.open(newline="", encoding="utf-8") as stream:
    reader = csv.DictReader(stream)
    if reader.fieldnames is None or not required.issubset(reader.fieldnames):
        raise RuntimeError("Manifest incompleto")
    rows = list(reader)

checkpoint = YOLO(BASE_MODEL)
coco_names = dict(checkpoint.names)
target_names = {0: "person", 1: "bicycle", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
if len(coco_names) != 80 or any(coco_names[class_id] != name for class_id, name in target_names.items()):
    raise RuntimeError("Il checkpoint non espone l'inventario COCO atteso")

seen_ids = set()
groups = defaultdict(list)
class_counts = Counter()
for row in rows:
    if not row["image_id"] or row["image_id"] in seen_ids or not row["group_id"]:
        raise RuntimeError("Identità immagine o gruppo non valida")
    seen_ids.add(row["image_id"])
    image_relative = Path(row["image_path"])
    image = (SOURCE / image_relative).resolve()
    if image_relative.is_absolute() or not image.is_relative_to(SOURCE) or not image.is_file() or image.suffix.lower() != ".jpg":
        raise RuntimeError(f"Percorso immagine non valido: {row['image_id']}")
    row["absolute_image_path"] = str(image)
    label_relative = row["label_path"]
    if row["polarity"] == "positive" and not label_relative:
        raise RuntimeError(f"Label positiva assente: {row['image_id']}")
    if label_relative:
        label_path = (SOURCE / label_relative).resolve()
        if Path(label_relative).is_absolute() or not label_path.is_relative_to(SOURCE) or not label_path.is_file():
            raise RuntimeError(f"Percorso label non valido: {row['image_id']}")
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parts = line.split()
            if len(parts) != 5:
                raise RuntimeError(f"Label YOLO non valida: {row['image_id']}")
            class_id = int(parts[0])
            center_x, center_y, width, height = map(float, parts[1:])
            if class_id not in target_names or not (0 <= center_x <= 1 and 0 <= center_y <= 1 and 0 < width <= 1 and 0 < height <= 1):
                raise RuntimeError(f"Box fuori contratto: {row['image_id']}")
            class_counts[class_id] += 1
    groups[row["group_id"]].append(row)

ordered_groups = sorted(
    groups.items(),
    key=lambda item: hashlib.sha256(f"phenocam-yolo-v2:{SEED}:{item[0]}".encode()).digest(),
)
validation_target = round(len(rows) * VALIDATION_FRACTION)
validation_rows = []
for _, group_rows in ordered_groups:
    if len(validation_rows) >= validation_target:
        break
    validation_rows.extend(group_rows)
validation_ids = {row["image_id"] for row in validation_rows}
training_rows = [row for row in rows if row["image_id"] not in validation_ids]
if {row["group_id"] for row in training_rows} & {row["group_id"] for row in validation_rows}:
    raise RuntimeError("Leakage di gruppo tra train e validation")

split_counts = {}
for split_name, split_rows in (("train", training_rows), ("val", validation_rows)):
    counts = Counter()
    for row in split_rows:
        if row["label_path"]:
            for line in (SOURCE / row["label_path"]).read_text(encoding="utf-8").splitlines():
                counts[int(line.split()[0])] += 1
    if any(counts[class_id] == 0 for class_id in target_names):
        raise RuntimeError(f"Classe operativa assente dallo split {split_name}")
    split_counts[split_name] = dict(sorted(counts.items()))

train_list = WORK / "train.txt"
validation_list = WORK / "validation.txt"
train_list.write_text("\n".join(row["absolute_image_path"] for row in training_rows) + "\n", encoding="utf-8")
validation_list.write_text("\n".join(row["absolute_image_path"] for row in validation_rows) + "\n", encoding="utf-8")
data_yaml = WORK / "dataset-v2.yaml"
data_yaml.write_text(yaml.safe_dump({"train": str(train_list), "val": str(validation_list), "names": coco_names}, sort_keys=False), encoding="utf-8")
audit = {"seed": SEED, "train_images": len(training_rows), "validation_images": len(validation_rows), "groups": len(groups), "class_instances": split_counts}
(WORK / "split-audit.json").write_text(json.dumps(audit, indent=2) + "\n", encoding="utf-8")
print(json.dumps(audit, indent=2))

## Training v2

`workers=0` evita processi figli fragili nei kernel macOS. Il batch 8 è conservativo per i 24 GB unificati e per il MacBook Air senza ventola. Il best checkpoint, non l'ultimo, diventa `models/yolo26n-v2.pt`. Lo split è riproducibile; PyTorch segnala però che una backward operation MPS non è bit-per-bit deterministica anche con `deterministic=True`.

In [ ]:
model = YOLO(BASE_MODEL)
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    patience=8,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=0,
    cache=False,
    seed=SEED,
    deterministic=True,
    project=str(RUNS),
    name="yolo26n-v2",
    exist_ok=True,
)
best_checkpoint = Path(model.trainer.best).resolve()
if not best_checkpoint.is_file():
    raise RuntimeError("Il training non ha prodotto best.pt")
shutil.copy2(best_checkpoint, V2_PT)
trained = YOLO(V2_PT)
if dict(trained.names) != coco_names:
    raise RuntimeError("Il modello v2 non conserva le 80 classi COCO")
print(f"Checkpoint v2: {V2_PT} ({V2_PT.stat().st_size / 1024 / 1024:.1f} MiB)")

## Metriche sul validation split

Il checkpoint base e v2 vengono misurati sullo stesso split. Il delta positivo indica un miglioramento di v2.

In [ ]:
baseline_metrics = YOLO(BASE_MODEL).val(
    data=str(data_yaml), split="val", imgsz=IMAGE_SIZE, batch=BATCH,
    device=DEVICE, workers=0, plots=True, project=str(RUNS), name="validation-base", exist_ok=True,
)
metrics = trained.val(
    data=str(data_yaml), split="val", imgsz=IMAGE_SIZE, batch=BATCH,
    device=DEVICE, workers=0, plots=True, project=str(RUNS), name="validation-pt", exist_ok=True,
)
metric_attributes = {"mAP50-95": "map", "mAP50": "map50", "precision": "mp", "recall": "mr"}
baseline_summary = {name: float(getattr(baseline_metrics.box, attribute)) for name, attribute in metric_attributes.items()}
v2_summary = {name: float(getattr(metrics.box, attribute)) for name, attribute in metric_attributes.items()}
summary = {
    "models": {"baseline": BASE_MODEL.name, "v2": V2_PT.name},
    "images": len(validation_rows),
    "baseline": baseline_summary,
    "v2": v2_summary,
    "delta_v2_minus_baseline": {name: v2_summary[name] - baseline_summary[name] for name in v2_summary},
    "v2_per_class_mAP50-95": {target_names[class_id]: float(metrics.box.maps[class_id]) for class_id in target_names},
}
(WORK / "metrics-v2.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2))

## Export ONNX e prova del runtime

L'export usa lo stesso formato statico del modello incluso. Il test seguente controlla metadati e forma end-to-end, poi esegue le sedici viste del runtime sullo stesso frame validation con base e v2. Le due annotazioni restano in `output/training-v2/` per il confronto visivo.

In [ ]:
exported = Path(trained.export(format="onnx", opset=20, imgsz=IMAGE_SIZE, batch=1, dynamic=False)).resolve()
if exported != V2_ONNX.resolve():
    shutil.copy2(exported, V2_ONNX)

from phenocam.classes.selection import enabled_class_names, model_class_ids
from phenocam.inference.pipeline import process_image
from phenocam.inference.runtime import create_session, model_contract

session = create_session(V2_ONNX)
_, _, width, height, onnx_names = model_contract(session)
model_class_ids(onnx_names, enabled_class_names())
if onnx_names != coco_names or (width, height) != (IMAGE_SIZE, IMAGE_SIZE):
    raise RuntimeError("Contratto ONNX v2 inatteso")
sample_input = Path(validation_rows[0]["absolute_image_path"])
sample_output = WORK / "runtime-sample.jpg"
inference_seconds = process_image(V2_ONNX, sample_input, sample_output, None)
base_sample_output = WORK / "runtime-sample-base.jpg"
base_inference_seconds = process_image(ROOT / "models" / "yolo26n.onnx", sample_input, base_sample_output, None)
if not sample_output.is_file():
    raise RuntimeError("Il runtime non ha prodotto l'immagine di test")
print({"onnx": str(V2_ONNX), "v2_sample": str(sample_output), "base_sample": str(base_sample_output), "v2_seconds": round(inference_seconds, 3), "base_seconds": round(base_inference_seconds, 3)})

## Lettura del risultato

Conserva v2 soltanto se le metriche sullo split fissato sono adeguate e il campione runtime è plausibile. Prima di sostituire il modello distribuito servono immagini operative escluse da questo dataset, soglie di accettazione esplicite e un confronto v1/v2 su recall e falsi positivi.

## Grafico conclusivo

A sinistra, valori più alti indicano una detection migliore; la linea verticale identifica l'epoch del checkpoint conservato. A destra, loss in calo e curve train/validation vicine indicano un apprendimento più stabile.

In [ ]:
import matplotlib.pyplot as plt

history_path = RUNS / "yolo26n-v2" / "results.csv"
with history_path.open(newline="", encoding="utf-8") as stream:
    history = list(csv.DictReader(stream))
if not history:
    raise RuntimeError("Cronologia del training assente")

epochs = [int(row["epoch"]) for row in history]
map50 = [float(row["metrics/mAP50(B)"]) for row in history]
map_all = [float(row["metrics/mAP50-95(B)"]) for row in history]
best_index = max(range(len(history)), key=map_all.__getitem__)
best_epoch = epochs[best_index]

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, map50, label="mAP50", linewidth=2)
axes[0].plot(epochs, map_all, label="mAP50–95", linewidth=2)
axes[0].axvline(best_epoch, color="0.4", linestyle="--", linewidth=1, label=f"best: epoch {best_epoch}")
axes[0].scatter([best_epoch], [map_all[best_index]], zorder=3)
axes[0].set(title="Qualità sul validation split", xlabel="Epoch", ylabel="mAP", ylim=(0, 1))
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(epochs, [float(row["train/box_loss"]) for row in history], label="box train", linewidth=2)
axes[1].plot(epochs, [float(row["val/box_loss"]) for row in history], label="box validation", linewidth=2, linestyle="--")
axes[1].plot(epochs, [float(row["train/cls_loss"]) for row in history], label="class train", linewidth=2)
axes[1].plot(epochs, [float(row["val/cls_loss"]) for row in history], label="class validation", linewidth=2, linestyle="--")
axes[1].set(title="Loss train e validation", xlabel="Epoch", ylabel="Loss")
axes[1].legend()
axes[1].grid(alpha=0.2)

figure.suptitle("YOLO26n v2 — andamento del training")
figure.tight_layout()
chart_path = WORK / "training-curves.png"
figure.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Grafico salvato in {chart_path}")